In [9]:
import numpy
from numpy import log
import torch
from torch import nn, Tensor
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
from torch.utils.data import Dataset, random_split
from torch.nn.utils.rnn import pad_sequence
import json
import random
import math

from sklearn.preprocessing import StandardScaler

#For cutting and increase the Median of the timestamps
THRESHOLD_TIMESTAMPS = 16

#Part of TRACE paper, after to cut if the sequence_lenght of the session is more bigger cut, if less padd
L_SEQUENCE_LENGHT = 64

#The learning embeddings for the Transformer_encoder_layer
EMBEDDING_DIM = 32

EPOCHS = 10

TRAINING_DATA_SET = 0.98
#DEV_SET  = 0.01
TEST_SET = 0.02

In [10]:
def extract_json(filename: str):
    """
    Extracts the json that are used in the project (OttoDataset) "train.jsonl"
    - input -> filename : the name of the session
    - return all the sessions
    """
    with open(filename, "r") as f:
        for line in f:
            session = json.loads(line)
            yield session["session"], session["events"]

In [11]:
sessions_bf_threshold = []

for i, (session_id, eventstotal) in enumerate(extract_json("../train.jsonl")):
    aids, timestamps, events_type = [], [], []
    for event in eventstotal:
        aids.append(event["aid"])
        timestamps.append(event["ts"])
        events_type.append(event["type"])
        
    sessions_bf_threshold.append({
            "session_id": i,
            "events": {
            "aid": aids,
            "timestamps": timestamps,
            "events_type": events_type    
            },
        })
    if i >= 200:
        break

In [12]:
class OttoDataSetSession(Dataset):
    """
    The first Dataset, before the TRACE Paper Cut Threshold of the Timestamps which in this case is 16 (THRESHOLD_TIMESTAMPS) 
    It Transforms the click aids in numbers 
    """
    def __init__(self, session):
        self.session = session
        self.event_map = {"clicks":1, "carts": 2, "orders": 3}

    def __len__(self) -> int:
        return len(self.session)

    def __getitem__(self, index):
        session = self.session[index]
                 
        events = session["events"]
        
        aids = torch.tensor(events["aid"], dtype=torch.int64)
        
        timestamps = torch.tensor(events["timestamps"], dtype=torch.long)
        
        events_type = torch.tensor( [self.event_map[e] for e in events["events_type"]], dtype=torch.int64)
        return {
            "session_id": torch.tensor(session["session_id"], dtype=torch.int64),
            "aid": aids,
            "timestamps": timestamps,
            "type": events_type
        }
    

In [ ]:
sessions_in_dataset = OttoDataSetSession(sessions_bf_threshold)
print(f"Total len of the Sessions: {len(sessions_in_dataset)}")

session_sample_lenght_after_threshold = []
for i in range(len(sessions_in_dataset)):
    sample = sessions_in_dataset[i]
    if len(sample["timestamps"]) >= THRESHOLD_TIMESTAMPS:
        session_sample_lenght_after_threshold.append(sample)

#print(np.mean(session_sample_lenght_after_threshold))
#print(f"The total of the median is for this total of {len(session_sample_lenght_after_threshold)} {np.median(session_sample_lenght_after_threshold)}")

Total len of the Sessions: 201


In [14]:

def most_frequent(List):
    """
    Counts the highest number of timestamps per product, used in Logit SAT4 (Seeing the same Aid 4 times)
    """
    dict = {}
    count, itm = 0, ''
    for item in reversed(List):
        dict[item] = dict.get(item, 0) + 1
        if dict[item] >= count :
            count, itm = dict[item], item
    return(count, itm)


def most_frequent(List):
    """
    Counts the highest number of timestamps per product, used in Logit SAT4 (Seeing the same Aid 4 times)
    """
    dict = {}
    count, itm = 0, ''
    for item in reversed(List):
        dict[item] = dict.get(item, 0) + 1
        if dict[item] >= count :
            count, itm = dict[item], item
    return(count, itm)

def custom_collate(batch):
    """
    aids = [
    torch.tensor([aid_to_idx[a] for a in d["aid"]], dtype=torch.long)
    for d in batch
    ]
    """
    
    inputs = [sample["inputs"] for sample in batch]
    targets = [sample["targets"] for sample in batch]
    
    
    aids = [torch.tensor(d["aid"], dtype=torch.long) for d in inputs]
    timestamps = [torch.tensor(d["timestamps"], dtype=torch.long) for d in inputs]
    events_type = [torch.tensor(d["type"], dtype=torch.int32) for d in inputs]
    
    
    
    aids_padded_input = pad_sequence(aids, batch_first=True, padding_value=0)
    timestamps_padded_input = pad_sequence(timestamps, batch_first=True, padding_value=0)
    events_type_padded_input = pad_sequence(events_type, batch_first=True, padding_value=0)
   
    session_id_inputs = [torch.tensor(d["session_id"], dtype=torch.int64) for d in inputs]
    
    return {
        "inputs" :{
        "aid" : aids_padded_input,
        "timestamps" : timestamps_padded_input,
        "type" : events_type_padded_input,
        "session_id" : session_id_inputs    
        },
        "target" : targets
        }



In [40]:
class CutOttoDataSet(OttoDataSetSession):
    def __init__(self, session):
        super().__init__(session)
        self.inputs_part, self.target_part = self.__cut_input_target__()
        self.input = self.inputs_part
        self.train_input = self.__sequenceLenght__()
        self.target = self.target_part 
        self.atc = self.__ATC_task_logit__()
    
    def __getitem__(self, index):
        inputs_sessions = {
            "session_id": self.train_input[index]["session_id"],
            "aid": self.train_input[index]["aid"],
            "timestamps": self.train_input[index]["timestamps"],
            "type": self.train_input[index]["type"],
        }

        #4 classes
        target_sessions = {
            "ATC": self.atc[index] 
            }

        return {
            "inputs": inputs_sessions,
            "targets": target_sessions
        }
     
        
    
    def __padding__(self, session):   
        padd_len = L_SEQUENCE_LENGHT - len(session["timestamps"])
        zeros = torch.zeros(padd_len, dtype=session["aid"].dtype)
        
        aid_padded = torch.concat((session["aid"], zeros))
        timestamps_padded = torch.concat((session["timestamps"], zeros))
        type_padded = torch.concat((session["type"], zeros))
        return {
            "session_id":session["session_id"],
            "aid": aid_padded,
            "timestamps": timestamps_padded,
            "type": type_padded
        }
           
        
    def __cut_input_target__(self, min_value=0.80, max_value=0.90):
        input_batches = []
        target_batches = []

        for single_session in self.session:
            cutting_number = random.uniform(min_value, max_value)

            
            n_events = len(single_session["timestamps"])
            input_size = int(n_events * cutting_number)

            input_part = {
                "session_id": single_session["session_id"],
                "aid": single_session["aid"][:input_size],
                "timestamps": single_session["timestamps"][:input_size],
                "type": single_session["type"][:input_size]
            }

            target_part = {
                "session_id": single_session["session_id"],
                "aid": single_session["aid"][input_size:],
                "timestamps": single_session["timestamps"][input_size:],
                "type": single_session["type"][input_size:]
            }

            input_batches.append(input_part)
            target_batches.append(target_part)

        return input_batches, target_batches


            
    def __sequenceLenght__(self):
        sessions_after_sequence_lenght = []
        for session in self.input:
            if len(session["timestamps"]) >= L_SEQUENCE_LENGHT:
                sessions_after_sequence_lenght.append({
                "session_id": session["session_id"],
                "aid": session["aid"][:L_SEQUENCE_LENGHT],
                "timestamps": session["timestamps"][:L_SEQUENCE_LENGHT],
                "type": session["type"][:L_SEQUENCE_LENGHT]
            })
            else: 
                sessions_after_sequence_lenght.append(self.__padding__(session))
        return sessions_after_sequence_lenght
    
    
    """
    Logits of the the model
    """
    def __ATC_task_logit__(self):
        """
        Labels for ATC (User added to the cart in the FUTURE)
        """
        

        logits_SAT = []
        for target_part in self.target:
            if 3 in target_part["type"]:
                logits_SAT.append(1)
            else:
                logits_SAT.append(0)

        return logits_SAT
    
    def __SAT__task_logit__(self):
        """
        Counts the highest number of timestamps per product, used in Logit SAT4 (Seeing the same Aid 4 times)
        """
        
        logits_SAT = []
        for session in self.target:
            AidsS_repeated = []
            count = 0    
            for aids in session["aid"]:
                AidsS_repeated.append(aids.item())
                count, product = most_frequent(AidsS_repeated)
            if count >= 4:
                logits_SAT.append(1)
            else:
                logits_SAT.append(0)
        return logits_SAT
    
    def __PD1_task_logit___(self):
        """
        Logits for PD1(Make any Purchase within a day)
        """
        logits_PD1 = []
        ONE_DAY = (86400 * 1000)
        for session in self.target:
            last_ts = session["timestamps"][-1].item()
            ordered_ts = session["timestamps"][session["type"] == 3]
            #Convert into int
            orders_ts = [ts.item() for ts in ordered_ts]
            
            is_purchase = any([(order <= last_ts + ONE_DAY) for order in orders_ts] )
            if is_purchase == True:
                logits_PD1.append(1)
            else:
                logits_PD1.append(0)
            
        return logits_PD1
    
    def __RA1_task_logit___(self):
        """
        Logits for RA1(Return to the same Aid in 1 days)
        """
        ONE_DAY = (86400 * 1000) 
        logits_RA1 = []
        for session in self.target:
            first_aid = session["aid"][0].item()
            first_ts = session["timestamps"][0].item()
            indices = [index for index, aid in enumerate(session["aid"]) if aid.item() == first_aid]
            other_ts_list = session["timestamps"][indices[1:]]

            is_aids = any((other_ts - first_ts) <= ONE_DAY for other_ts in other_ts_list)
            
            if is_aids == True:
                logits_RA1.append(1)
            else: 
                logits_RA1.append(0)
        return logits_RA1
            
           


In [41]:
#DataSet    
data_set_after_L = CutOttoDataSet(session_sample_lenght_after_threshold)

      
#See how many lenghts are for the Aids in the raining batch and testing batch

    


   
print(len(data_set_after_L.__sequenceLenght__()[0]["timestamps"]))
    
print("================================================ (Logits part) ===================================================")
print("Logits for the ATC (Add to the Cart)")
print(data_set_after_L.__ATC_task_logit__())
    
print("Logits for SAT4 (Seeing the same Aid 4 times)")
print(data_set_after_L.__SAT__task_logit__())
    
print("Logits for PD1 (Make any Purchase within a day)")
print(data_set_after_L.__PD1_task_logit___())
    
print("Logits for RA1 (Return to the same Aid in 1 days)")
print(data_set_after_L.__RA1_task_logit___())

64
================================================ (Logits part) ===================================================
Logits for the ATC (Add to the Cart)
[1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Logits for SAT4 (Seeing the same Aid 4 times)
[0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0,

In [42]:

scaler_time_elapsed = StandardScaler()
scaler_time_between = StandardScaler()

#Categorical Embeddings
aids_session = []
event_session = []

#Time Embeddings
log_delta_elapsed = []
log_between_time = []

for sample in data_set_after_L:
    single_session = sample["inputs"]
    aids_session.extend(single_session["aid"].tolist())
    event_session.extend(single_session["type"].tolist())
    
    
    
    ts_last = single_session["timestamps"][-1]
    ts_first = single_session["timestamps"][0]
    
    log_delta_elapsed.append(log(1 +  (ts_last.item() - ts_first.item())))

    
    deltas_this_session = []
    
    for j in range(len(single_session["timestamps"]) - 1):
        delta_between_times = (single_session["timestamps"][j+1].item() - single_session["timestamps"][j].item())
        deltas_this_session.append(log(1 + delta_between_times))
    log_between_time.append(deltas_this_session)
    


#AID Embeddings (Categorical, Embedding := 32)
aid_vocab = sorted(set(aids_session))
print(aids_session)
#aid_to_idx = {aid: i for i, aid in enumerate(aid_vocab)}
num_embeddings_aid = len(aid_vocab)

#Type Event Embeddings(Categorical, Embedding := 32)
type_event_vocab = sorted(set(event_session))
num_embeddings_event_type = len(type_event_vocab)

#Time Embeddings(Timestamps, Embedding := 1 + +1)
array_time_delta_elapsed = numpy.array(log_delta_elapsed).reshape(-1, 1)
standard_time_delta_elapsed = scaler_time_elapsed.fit_transform(array_time_delta_elapsed)

flat_time_between = [d for session_list in log_between_time for d in session_list]
array_time_between = numpy.array(flat_time_between).reshape(-1, 1)
standard_time_between = scaler_time_between.fit_transform(array_time_between)

number_session = 0
session_standard_time_between = []
for session_list in log_between_time:
    L = len(session_list)
    session_standard_time_between.append(standard_time_between[number_session : number_session + L].flatten().tolist())
    number_session += L


[1517085, 1563459, 1309446, 16246, 1781822, 1152674, 1649869, 461689, 305831, 461689, 362233, 1649869, 1649869, 984597, 1649869, 803544, 1110941, 1190046, 1760685, 631008, 461689, 1190046, 1650637, 313546, 1650637, 979517, 351157, 1062149, 1157384, 1841388, 1469630, 305831, 1110548, 1110548, 305831, 1650114, 1604396, 1009750, 1800933, 495779, 394655, 495779, 789245, 789245, 366890, 361317, 1700164, 1755597, 789245, 784978, 1171505, 784978, 1700164, 784978, 1521766, 1725503, 528847, 1816325, 984597, 1072782, 173702, 1072782, 1407538, 1629651, 424964, 1492293, 1492293, 910862, 910862, 1491172, 1491172, 424964, 1515526, 440486, 109488, 1507622, 1734061, 854637, 854637, 718983, 215311, 215311, 718983, 711125, 711125, 50049, 105393, 105393, 959544, 1734061, 1842593, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 763743, 137492, 504789, 137492, 795863, 378348, 795863, 26638, 817441, 1405904, 545290, 935830, 935830, 1593105, 4276

/var/folders/64/_d7wbs111535lmdv26qfwp000000gn/T/ipykernel_3606/875477601.py:22: RuntimeWarning: invalid value encountered in log
  log_delta_elapsed.append(log(1 +  (ts_last.item() - ts_first.item())))
/var/folders/64/_d7wbs111535lmdv26qfwp000000gn/T/ipykernel_3606/875477601.py:29: RuntimeWarning: invalid value encountered in log
  deltas_this_session.append(log(1 + delta_between_times))


In [43]:
class MLP(nn.Module):
    def __init__(self, input_channels : int, output_channels : int):
        super().__init__()
        self.layers = nn.Sequential(
                nn.Linear(input_channels, 64),
                nn.ReLU(),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Linear(32, output_channels)
            )
    def forward(self, x):
            return self.layers(x)
        
        
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        self.d_model = d_model

        
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)  

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: Tensor) -> Tensor:
        # x: (B, L, d_model)
        x = x + self.pe[:, :x.size(1)]
        return x

In [51]:

class TRACE(nn.Module):
    def __init__(self, num_embeddings_aid : int, 
                 num_embeddings_event_type : int,
                 num_classes : int = 4
               ):
        super(TRACE, self).__init__()
        
        self.D_model = EMBEDDING_DIM * 2 + 2 # 66
        print(self.D_model)
        
        
        self.embedding_aid = nn.Embedding(num_embeddings=num_embeddings_aid,
                                          embedding_dim=EMBEDDING_DIM)
        
        self.embedding_eventtype = nn.Embedding(num_embeddings=num_embeddings_event_type,
                                                embedding_dim=EMBEDDING_DIM) 

        self.positional_embedding = PositionalEncoding(d_model=self.D_model)
        
        #D_model
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=self.D_model, 
                                                        nhead=6, # 8
                                                        dim_feedforward=128,
                                                        dropout=0.2,
                                                        activation="relu",
                                                        batch_first=True)
        
        self.encoder = nn.TransformerEncoder(encoder_layer=self.encoder_layer,
                                             num_layers=1)
        
                
        self.GBAP = nn.AdaptiveMaxPool1d(output_size=1)
        
        self.__ATC__ = MLP(input_channels=self.D_model, output_channels=1)
        
        """
        self.MLP_layer = MLP(input_channels=self.D_model, 
                             output_channels=num_classes)
        """
        
        
    def forward(self, aids_ids: Tensor, type_ids: Tensor, delta_elapsed : Tensor, delta_between : Tensor) -> Tensor:
        
        #Categorical Learning Embeddings: 32 + 32
        aid_emb = self.embedding_aid(aids_ids)
        type_emb = self.embedding_eventtype(type_ids)
        
        B, L, _ = aid_emb.shape #(Batch, Len_seque , 1)
        
        
        #Time Learning Embeddings: 1 + 1
        delta_between = delta_between.unsqueeze(-1)        
        delta_between = torch.cat([delta_between, delta_between[:, -1:, :]],dim=1)                                                  
        delta_elapsed = delta_elapsed.unsqueeze(-1).unsqueeze(-1)
        delta_elapsed = delta_elapsed.expand(-1, L, -1)
      

        x = torch.cat(
            [aid_emb,
            type_emb,
            delta_between,
            delta_elapsed],
            dim=-1)  
            
        
        positional_embed = self.positional_embedding(x)
        
        encoder = self.encoder(positional_embed)
        
        global_avarage_pooling = self.GBAP(encoder.transpose(1,2)).squeeze(-1)
              
        logits = self.__ATC__(global_avarage_pooling)
        
        return logits  

In [52]:

#DataLoaders

train_data, test_data = random_split(dataset=data_set_after_L, lengths=[TRAINING_DATA_SET,TEST_SET])


#DEV_SET
"""
validation_loader = DataLoader(
    dataset=val_data,
    batch_size=32,
    collate_fn=custom_collate,
    shuffle=False
)
"""
train_loader = DataLoader(
    dataset=train_data,
    batch_size=32,
    shuffle=True
)



test_loader = DataLoader(
    dataset=test_data,
    batch_size=32,
    shuffle=False
)


In [53]:

for batch_training in train_loader:
    print(batch_training)
    break
    sample = batch_training["inputs"]

    print(
        f"Shape Aids: {sample['aid']}, "
        f"Shape Timestamps: {sample['timestamps']}, "
        f"Shape Type: {sample['type']}"
    )
    break

  

{'inputs': {'session_id': tensor([103, 159,  47, 140, 194, 133,   2,  25,  73,  17,  41, 184, 101,  29,
         11,  48,  46, 120, 152,  91,  38,  23, 104,  76,  40, 161, 160,  52,
        175,  81, 183, 113]), 'aid': tensor([[1559521,  478456,  910710,  ..., 1455451, 1757395,  216337],
        [1795756, 1679611,  992771,  ...,  226483,  226483, 1558710],
        [ 387299, 1367632, 1162940,  ...,       0,       0,       0],
        ...,
        [ 645865,  872414,  502427,  ..., 1719820,  867165, 1398619],
        [ 168929, 1732105, 1732105,  ...,  177002,  177002,  177002],
        [ 721434,  721434,  269257,  ...,  875555, 1440782,  806489]]), 'timestamps': tensor([[1659304801508, 1659304843491, 1659304865343,  ...,
         1660862508096, 1660862586291, 1660862630789],
        [1659304802461, 1659304824475, 1659304847236,  ...,
         1660266235273, 1660392608066, 1660411496203],
        [1659304800685, 1659306203590, 1659306311078,  ...,
                     0,             0,    

In [54]:
max_aid = max(
    session["inputs"]["aid"].max().item()
    for session in data_set_after_L
)
max_type = max(
    session["inputs"]["type"].max().item()
    for session in data_set_after_L
)



num_embeddings_aid = max_aid + 1  
num_embeddings_event_type = max_type + 1
trace_model = TRACE(
    num_embeddings_aid=num_embeddings_aid,
    num_embeddings_event_type=num_embeddings_event_type,
    num_classes=4
)
"""print("MAX aid in batch:", max(aids))
print("Embedding size:", trace_model.embedding_aid.num_embeddings)
print("MAX type in batch:", len(events_type))
print("Embedding size:", trace_model.embedding_eventtype.num_embeddings)"""

66


'print("MAX aid in batch:", max(aids))\nprint("Embedding size:", trace_model.embedding_aid.num_embeddings)\nprint("MAX type in batch:", len(events_type))\nprint("Embedding size:", trace_model.embedding_eventtype.num_embeddings)'

In [61]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(trace_model.parameters(), lr=1e-5, weight_decay=1e-6)


for epoch in range(EPOCHS):
    trace_model.train()
    epoch_loss = 0.0
    
    for batch_trainer in train_loader:
        evidence = batch_trainer["inputs"]
        label_sample = batch_trainer["targets"]
        label = label_sample["ATC"].unsqueeze_(1)
        
            
        aids = evidence["aid"]
        events_type = evidence["type"]
        timestamps = evidence["timestamps"]      

        ts_first = timestamps[:, 0]
        ts_last = timestamps[:, -1]

        
        log_delta_elapsed = torch.log1p(torch.clamp(ts_last - ts_first, min=0))

        
        delta_elapsed = (log_delta_elapsed - log_delta_elapsed.mean()) / (log_delta_elapsed.std() + 1e-6)           # (B,)

    
        log_delta_between = torch.log1p(torch.clamp(timestamps[:, 1:] - timestamps[:, :-1], min=0))

        mean_between = log_delta_between.mean(dim=1, keepdim=True)
        std_between  = log_delta_between.std(dim=1, keepdim=True)

        delta_between = (log_delta_between - mean_between) / (std_between + 1e-6)                        

    
        pred = trace_model(
            aids,
            events_type,
            delta_elapsed,
            delta_between
        )

        loss = criterion(pred, label.float())
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    train_loss = epoch_loss/len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {epoch_loss/len(train_loader):.4f}")


Epoch [1/10], Loss: 0.6606
Epoch [2/10], Loss: 0.6592
Epoch [3/10], Loss: 0.6586
Epoch [4/10], Loss: 0.6564
Epoch [5/10], Loss: 0.6530
Epoch [6/10], Loss: 0.6541
Epoch [7/10], Loss: 0.6515
Epoch [8/10], Loss: 0.6505
Epoch [9/10], Loss: 0.6513
Epoch [10/10], Loss: 0.6471


In [321]:
import torch
import torch.nn as nn

# ====== Configuración ======
BATCH = 16
L = 48
EMBEDDING_DIM = 32
num_embeddings_aid = 5000
num_embeddings_event_type = 10
num_classes = 4

# ====== Modelo ======
trace_model = TRACE(
    num_embeddings_aid=num_embeddings_aid,
    num_embeddings_event_type=num_embeddings_event_type,
    num_classes=num_classes
)

criterion = nn.CrossEntropyLoss()

# ====== Mock batch CORRECTO ======
aids_ids = torch.randint(0, num_embeddings_aid, (BATCH, L))      # (B, L)
type_ids = torch.randint(0, num_embeddings_event_type, (BATCH, L))  # (B, L)

delta_elapsed = torch.randn(BATCH)       # ✅ (B,)
delta_between = torch.randn(BATCH, L-1)  # ✅ (B, L-1)

labels = torch.randint(0, num_classes, (BATCH,))  # (B,)

# ====== Forward ======
logits = trace_model(aids_ids, type_ids, delta_elapsed, delta_between)

print("Logits shape:", logits.shape)   # ✅ debe ser (BATCH, 4)

loss = criterion(logits, labels)
print("Loss:", loss.item())


66
torch.Size([16, 66])
Logits shape: torch.Size([16, 4])
Loss: 1.4645079374313354
